In [ ]:
# Annuity Calculator
# Computes the accumulated value of:
#   - Simple annuities (single fixed interest rate, entered via GUI)
#   - Block annuities  (multiple interest rate blocks, loaded from CSV)
#
# CSV format for block annuities — one row per block:
#   interest_rate_per_period, payment_per_period, number_of_periods
#
# A block with payment = 0 is valid and models a period where no new
# payments are made but the existing balance continues to earn interest.

import csv
import tkinter as tk
from tkinter import ttk, filedialog, messagebox


# ── Core calculation classes and functions ────────────────────────────────────

class Block:
    """Represents one block of a savings annuity with a fixed interest rate per period.

    Attributes:
        i  -- interest rate per period
        r  -- payment made at the end of each period
        n  -- number of periods in this block
        v  -- accumulated value (computed by acc_value / fin_value)
    """

    def __init__(self, i, r, n):
        self.i = i
        self.r = r
        self.n = n
        self.v = 0

    def acc_value(self):
        """Compute the accumulated value of this block at the end of its own term.

        Uses the ordinary annuity formula: v = R * [(1+i)^n - 1] / i
        Special case: when i = 0, the accumulated value is simply R * n.
        """
        if self.i == 0:
            self.v = self.r * self.n
        else:
            g = (1 + self.i) ** self.n
            self.v = self.r * (g - 1) / self.i
        return self.v

    def fin_value(self, i_new, n_new):
        """Compound this block's value forward through a subsequent block.

        After a block ends, its accumulated value continues to earn interest
        at the rates of all subsequent blocks. This method must be called once
        for each subsequent block.

        Args:
            i_new  -- interest rate per period of the subsequent block
            n_new  -- number of periods in the subsequent block
        """
        self.v = self.v * (1 + i_new) ** n_new
        return self.v


def compute_blocks(blocks):
    """Compute accumulated and final values for a list of Block objects.

    Returns:
        acc_values   -- list of each block's value at the end of its own term
        final_values -- list of each block's value at the end of the full term
        total        -- sum of all final values
    """
    n = len(blocks)
    V = [b.acc_value() for b in blocks]
    acc_values = list(V)

    # Compound each block forward through all subsequent blocks
    for k in range(n - 1):
        for j in range(k + 1, n):
            V[k] = blocks[k].fin_value(blocks[j].i, blocks[j].n)

    return acc_values, V, sum(V)


# ── GUI application ───────────────────────────────────────────────────────────

class AnnuityCalculatorApp:

    FREQ_OPTIONS = [
        ("Annual (1)",       1),
        ("Semi-annual (2)",  2),
        ("Quarterly (4)",    4),
        ("Monthly (12)",    12),
        ("Weekly (52)",     52),
        ("Daily (365)",    365),
    ]

    def __init__(self, root):
        self.root = root
        self.root.title("Annuity Calculator")
        self.root.resizable(False, False)

        notebook = ttk.Notebook(root)
        notebook.pack(fill="both", expand=True, padx=12, pady=12)

        simple_frame = ttk.Frame(notebook, padding=12)
        notebook.add(simple_frame, text="  Simple Annuity  ")
        self._build_simple_tab(simple_frame)

        block_frame = ttk.Frame(notebook, padding=12)
        notebook.add(block_frame, text="  Block Annuity  ")
        self._build_block_tab(block_frame)

    # ── Simple Annuity tab ────────────────────────────────────────────────────

    def _build_simple_tab(self, frame):
        # ── Inputs ──
        inp = ttk.LabelFrame(frame, text="Inputs", padding=12)
        inp.grid(row=0, column=0, sticky="ew", pady=(0, 10))

        ttk.Label(inp, text="Nominal Annual Rate (%):").grid(
            row=0, column=0, sticky="w", pady=5)
        self.s_rate = tk.StringVar()
        ttk.Entry(inp, textvariable=self.s_rate, width=20).grid(
            row=0, column=1, padx=12, pady=5)

        ttk.Label(inp, text="Compounding / Payment Frequency:").grid(
            row=1, column=0, sticky="w", pady=5)
        self.s_freq_val = tk.IntVar(value=12)
        freq_labels = [label for label, _ in self.FREQ_OPTIONS]
        self.s_freq_label = tk.StringVar(value="Monthly (12)")
        freq_cb = ttk.Combobox(inp, textvariable=self.s_freq_label,
                               values=freq_labels, state="readonly", width=18)
        freq_cb.grid(row=1, column=1, padx=12, pady=5)
        freq_cb.bind("<<ComboboxSelected>>", self._on_freq_changed)

        ttk.Label(inp, text="Payment per Period ($):").grid(
            row=2, column=0, sticky="w", pady=5)
        self.s_pmt = tk.StringVar()
        ttk.Entry(inp, textvariable=self.s_pmt, width=20).grid(
            row=2, column=1, padx=12, pady=5)

        ttk.Label(inp, text="Number of Periods:").grid(
            row=3, column=0, sticky="w", pady=5)
        self.s_nper = tk.StringVar()
        ttk.Entry(inp, textvariable=self.s_nper, width=20).grid(
            row=3, column=1, padx=12, pady=5)

        # ── Calculate button ──
        ttk.Button(frame, text="Calculate", command=self._calc_simple).grid(
            row=1, column=0, pady=8)

        # ── Result ──
        res = ttk.LabelFrame(frame, text="Result", padding=12)
        res.grid(row=2, column=0, sticky="ew")

        self.s_result = tk.StringVar(value="—")
        ttk.Label(res, textvariable=self.s_result,
                  font=("Arial", 13, "bold")).grid(row=0, column=0, pady=4)

        self.s_detail = tk.StringVar(value="")
        ttk.Label(res, textvariable=self.s_detail,
                  foreground="grey", font=("Arial", 9)).grid(row=1, column=0)

    def _on_freq_changed(self, _event):
        label = self.s_freq_label.get()
        for lbl, val in self.FREQ_OPTIONS:
            if lbl == label:
                self.s_freq_val.set(val)
                break

    def _calc_simple(self):
        try:
            annual_rate = float(self.s_rate.get()) / 100
            freq        = self.s_freq_val.get()
            pmt         = float(self.s_pmt.get())
            n           = float(self.s_nper.get())
        except ValueError:
            messagebox.showerror("Input Error", "Please enter valid numeric values.")
            return

        i = annual_rate / freq
        b = Block(i, pmt, n)
        total = b.acc_value()

        self.s_result.set(f"Accumulated Value:   ${total:,.2f}")
        self.s_detail.set(
            f"i = {i * 100:.6f}% per period   |   "
            f"n = {int(n)} periods   |   R = ${pmt:,.2f}"
        )

    # ── Block Annuity tab ─────────────────────────────────────────────────────

    def _build_block_tab(self, frame):
        # ── File selection ──
        fsel = ttk.LabelFrame(frame, text="CSV Input File", padding=12)
        fsel.grid(row=0, column=0, sticky="ew", pady=(0, 10))

        self.b_filepath = tk.StringVar()
        ttk.Entry(fsel, textvariable=self.b_filepath, width=44).grid(
            row=0, column=0, padx=(0, 8))
        ttk.Button(fsel, text="Browse…", command=self._browse_csv).grid(
            row=0, column=1)

        ttk.Label(fsel,
                  text="CSV format per row:  "
                       "interest_rate_per_period,  payment_per_period,  num_periods",
                  foreground="grey", font=("Arial", 8)).grid(
            row=1, column=0, columnspan=2, sticky="w", pady=(8, 0))

        # ── Calculate button ──
        ttk.Button(frame, text="Calculate", command=self._calc_block).grid(
            row=1, column=0, pady=8)

        # ── Results ──
        res = ttk.LabelFrame(frame, text="Results", padding=12)
        res.grid(row=2, column=0, sticky="nsew")

        self.b_results = tk.Text(
            res, height=16, width=58,
            font=("Courier New", 10), state="disabled",
            bg="#f8f8f8", relief="flat")
        self.b_results.grid(row=0, column=0)

        sb = ttk.Scrollbar(res, command=self.b_results.yview)
        sb.grid(row=0, column=1, sticky="ns")
        self.b_results.config(yscrollcommand=sb.set)

    def _browse_csv(self):
        path = filedialog.askopenfilename(
            title="Select block annuity CSV file",
            filetypes=[("CSV files", "*.csv"), ("All files", "*.*")])
        if path:
            self.b_filepath.set(path)

    def _calc_block(self):
        path = self.b_filepath.get().strip()
        if not path:
            messagebox.showerror("Input Error", "Please select a CSV file.")
            return

        try:
            B = []
            with open(path) as f:
                for row in csv.reader(f):
                    if row:
                        B.append(Block(float(row[0]), float(row[1]), float(row[2])))
        except Exception as e:
            messagebox.showerror("File Error", f"Could not read CSV file:\n{e}")
            return

        if not B:
            messagebox.showerror("File Error", "CSV file is empty.")
            return

        acc_vals, final_vals, total = compute_blocks(B)

        lines = []
        lines.append(f"{'Block':<7} {'Rate/Period':>13} {'Payment':>12} {'Periods':>8}")
        lines.append("─" * 44)
        for k, b in enumerate(B):
            pmt_str = f"${b.r:>10,.2f}" if b.r > 0 else f"{'—':>11}"
            lines.append(
                f"{k:<7} {b.i * 100:>12.6f}%  {pmt_str}  {int(b.n):>6}")

        lines.append("")
        lines.append("Accumulated value at end of each block:")
        for k, v in enumerate(acc_vals):
            lines.append(f"  Block {k}:  ${v:>14,.2f}")

        if len(B) > 1:
            lines.append("")
            lines.append("Value of each block at end of full term:")
            for k, v in enumerate(final_vals):
                lines.append(f"  Block {k}:  ${v:>14,.2f}")

        lines.append("")
        lines.append("─" * 44)
        lines.append(f"  Total:    ${total:>14,.2f}")

        self._set_block_results("\n".join(lines))

    def _set_block_results(self, text):
        self.b_results.config(state="normal")
        self.b_results.delete("1.0", tk.END)
        self.b_results.insert(tk.END, text)
        self.b_results.config(state="disabled")


# ── Entry point ───────────────────────────────────────────────────────────────

if __name__ == "__main__":
    root = tk.Tk()
    AnnuityCalculatorApp(root)
    root.mainloop()
